In [1]:
!pip -q install transformers datasets evaluate accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments, Trainer,
    EarlyStoppingCallback
)

BASE_MODEL = "mangsense/codebert_java" 
data_files = {
    "train": "/kaggle/input/dataset/train.jsonl",
    "validation": "/kaggle/input/dataset/valid.jsonl",
    "test": "/kaggle/input/dataset/test.jsonl",
}

2026-01-29 05:32:34.623108: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769664754.816937      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769664754.878126      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769664755.349488      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769664755.349528      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769664755.349531      55 computation_placer.cc:177] computation placer alr

In [3]:
ds = load_dataset("json", data_files=data_files)  # load JSON/JSONL rất tiện :contentReference[oaicite:1]{index=1}

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [4]:
def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# bỏ các cột metadata khỏi input của model để train nhanh (vẫn giữ labels)
keep = {"labels"}
remove_cols = [c for c in ds["train"].column_names if c not in keep]
ds_tok = ds.map(tok, batched=True, remove_columns=remove_cols)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

acc = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }


Map:   0%|          | 0/788 [00:00<?, ? examples/s]

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

Map:   0%|          | 0/84 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [7]:
import transformers
print(transformers.__version__)


4.57.1


In [11]:
import transformers
print("transformers =", transformers.__version__)  

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

HUB_MODEL_ID = "khanhtran0111/codebert-java-vul4j-ft"

args = TrainingArguments(
    output_dir="out",
    eval_strategy="epoch",         
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
test_metrics = trainer.evaluate(ds_tok["test"])
print("Test metrics:", test_metrics)
trainer.push_to_hub()


transformers = 4.57.1


/tmp/ipykernel_55/4269381982.py:43: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.427839,0.835294,0.300000
2,No log,0.441472,0.847059,0.000000
3,No log,0.558748,0.788235,0.250000


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Test metrics: {'eval_loss': 0.4424135684967041, 'eval_accuracy': 0.7976190476190477, 'eval_f1': 0.10526315789473684, 'eval_runtime': 1.426, 'eval_samples_per_second': 58.906, 'eval_steps_per_second': 2.104, 'epoch': 3.0}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/khanhtran0111/codebert-java-vul4j-ft/commit/0c3833560ad811ed9cb54be68e814bce7efe15cc', commit_message='End of training', commit_description='', oid='0c3833560ad811ed9cb54be68e814bce7efe15cc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/khanhtran0111/codebert-java-vul4j-ft', endpoint='https://huggingface.co', repo_type='model', repo_id='khanhtran0111/codebert-java-vul4j-ft'), pr_revision=None, pr_num=None)